<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

## **Proyecto Random Forest**

**Autora:** Anais Aponte  
**Bootcamp:** 4Geeks Academy – Intro to Machine Learning  
**Proyecto:** Proyecto Random Forest  

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### 📝 **Instrucciones**

#### **Prediciendo la diabetes**

**Objetivo**: En este proyecto el objetivo es mejorar el modelo realizado en el proyecto de arbol de decision para aumentar el accuracy. 
</div>

----

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### 📝 **Paso 1: Carga del conjunto de datos**

El dataset utilizado en este proyecto es **diabetes.csv**.

Para facilitar la reproducibilidad y evitar dependencias externas, el archivo ha sido previamente descargado y almacenado dentro del repositorio en la siguiente ruta:

📁 `data/raw/diabetes.csv`

De esta forma, el notebook puede ejecutarse directamente sin necesidad de acceder a enlaces externos.

A continuación se realizan primero las importaciones necesarias y, posteriormente, la carga del dataset para iniciar el análisis.

</div>

In [21]:
# Manipulación de datos
import pandas as pd
import numpy as np
# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
# Preprocesamiento
from sklearn.impute import SimpleImputer
# División de datos
from sklearn.model_selection import train_test_split, GridSearchCV
# Modelos
from sklearn.ensemble import RandomForestClassifier
# Métricas
from sklearn.metrics import make_scorer, recall_score, classification_report
# Importamos la librería para guardar modelos
import joblib


In [3]:
# CARGAMOS EL FICHERO CON LOS DATOS A ANALIZAR
df = pd.read_csv('../data/raw/diabetes.csv')

# Visualizamos 10 registros aleatorios del dataset con sample(). Esto nos permite observar una muestra representativa de los datos,
# ya que algunos datasets pueden estar ordenados (por fecha, id, etc.).
df.sample(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
195,5,158,84,41,210,39.4,0.395,29,1
194,8,85,55,20,0,24.4,0.136,42,0
621,2,92,76,20,0,24.2,1.698,28,0
551,3,84,68,30,106,31.9,0.591,25,0
54,7,150,66,42,342,34.7,0.718,42,0
200,0,113,80,16,0,31.0,0.874,21,0
626,0,125,68,0,0,24.7,0.206,21,0
181,0,119,64,18,92,34.9,0.725,23,0
64,7,114,66,0,0,32.8,0.258,42,1
504,3,96,78,39,0,37.3,0.238,40,0


<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

#### **Paso 1.1: Limpiar nulos (ceros)**

</div>

In [4]:
# Contar ceros por columna en todo el dataset
ceros = (df == 0).sum()

# Convertir a DataFrame para verlo mejor
ceros_df = ceros.to_frame(name='num_ceros')

# Añadir porcentaje
ceros_df['porcentaje'] = (ceros_df['num_ceros'] / len(df)) * 100

# Ordenar de mayor a menor
ceros_df = ceros_df.sort_values(by='num_ceros', ascending=False)

ceros_df

,num_ceros,porcentaje
Outcome,500,65.104167
Insulin,374,48.697917
SkinThickness,227,29.557292
Pregnancies,111,14.453125
BloodPressure,35,4.557292
BMI,11,1.432292
Glucose,5,0.651042
DiabetesPedigreeFunction,0,0.000000
Age,0,0.000000


In [5]:
# Creamos una copia del dataset para trabajar sin modificar el original
df_sinceros = df.copy()

In [6]:
# Reemplazamos los valores 0 por NaN en variables donde no son fisiológicamente posibles
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_with_zeros:
    df_sinceros[col] = df_sinceros[col].replace(0, np.nan)

In [7]:
# Imputamos los valores faltantes utilizando la mediana para evitar sesgos por outliers

imputer = SimpleImputer(strategy='median')
df_sinceros[cols_with_zeros] = imputer.fit_transform(df_sinceros[cols_with_zeros])


In [8]:
# Verificamos el número de ceros en todas las variables tras el tratamiento
ceros_sinceros = (df_sinceros == 0).sum()

ceros_sinceros

Pregnancies                 111
Glucose                       0
BloodPressure                 0
SkinThickness                 0
Insulin                       0
BMI                           0
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

#### **Paso 1.2: Separar X y y**

</div>

In [9]:
# Separamos variables predictoras (X) y variable objetivo (y)
X = df_sinceros.drop(columns='Outcome')
y = df_sinceros['Outcome']

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

#### **Paso 1.3: Train / Test split**

</div>

In [10]:
# Dividimos los datos en entrenamiento y test manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,1,90.0,62.0,12.0,43.0,27.2,0.580,24
711,5,126.0,78.0,27.0,22.0,29.6,0.439,40
373,2,105.0,58.0,40.0,94.0,34.9,0.225,25
46,1,146.0,56.0,29.0,125.0,29.7,0.564,29
682,0,95.0,64.0,39.0,105.0,44.6,0.366,22


--------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2: Construye un random forest (modelo base)**

</div>

In [11]:
# Entrenamos el modelo con los valores por defecto a saber: 
# n_estimators = 100
# max_depth = None → crece todo lo que pueda
# min_samples_split = 2
# min_samples_leaf = 1
# max_features = "sqrt" (en clasificación)
# bootstrap = True

model_base = RandomForestClassifier(random_state = 42)

model_base.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.1: Predecimos y medimos el modelo base**

</div>

In [16]:
# Generamos las predicciones del modelo base sobre el conjunto de test
y_pred_base = model_base.predict(X_test)

In [28]:
# Mostrar métricas del modelo base de RF
print("metricas modelo RF base:")
print(classification_report(y_test, y_pred_base))

metricas modelo RF base:
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       100
           1       0.73      0.59      0.65        54

    accuracy                           0.78       154
   macro avg       0.76      0.74      0.75       154
weighted avg       0.77      0.78      0.77       154



<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### 💡 **Observación**

El modelo base de Random Forest alcanza un rendimiento muy similar, pero no supera al obtenido con el árbol de decisión ajustado con `criterion='entropy'`. Esto sugiere que, para este dataset, el uso de un ensemble no supone una mejora clara en términos de accuracy frente a un árbol individual bien configurado.

Aun así, Random Forest sigue siendo una alternativa interesante por su capacidad para generar predicciones más estables y por su menor sensibilidad al sobreajuste en muchos problemas.

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2.2: Probar con hiperparametros**

</div>

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### 💡 **Nota:**

En esta fase se amplía la exploración de hiperparámetros utilizando GridSearchCV, permitiendo evaluar múltiples combinaciones de forma sistemática mediante validación cruzada.

Además de los hiperparámetros básicos del modelo (como n_estimators y bootstrap), se incluyen otros parámetros relevantes como max_depth y min_samples_split, con el objetivo de optimizar tambien los arboles que componen al bosque.

Dado el contexto del problema, se prioriza el recall como métrica de evaluación, buscando maximizar la detección de casos positivos.

</div>

In [14]:
# Definimos el scorer priorizando el recall (contexto médico)
recall_scorer = make_scorer(recall_score)

# Definimos el modelo base
rf = RandomForestClassifier(random_state=42)

# Definimos la rejilla de hiperparámetros a explorar
param_grid = {
    'n_estimators': [50, 100, 200],        # Nº de árboles
    'max_depth': [5, 10, None],            # Profundidad de los árboles
    'min_samples_split': [2, 5, 10],       # Mínimo para dividir nodo
    'bootstrap': [True, False]             # Muestreo con reemplazo
}

# Configuramos GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=recall_scorer,  # 🔥 MUY IMPORTANTE (no accuracy)
    cv=5,
    n_jobs=-1
)

# Entrenamos
grid_search.fit(X_train, y_train)

# Mejores hiperparámetros
print("Mejores hiperparámetros:", grid_search.best_params_)

# Mejor score
print("Mejor recall en CV:", grid_search.best_score_)

# Mejor modelo
best_rf = grid_search.best_estimator_

Mejores hiperparámetros: {'bootstrap': False, 'max_depth': None, 'min_samples_split': 10, 'n_estimators': 100}
Mejor recall en CV: 0.6124031007751938


In [29]:
# Predicciones del mejor modelo encontrado por GridSearch
y_pred_grid = best_rf.predict(X_test)

# Mostrar métricas modelo grid
print("metricas modelo RF GridSearch:")
print(classification_report(y_test, y_pred_grid))

metricas modelo RF GridSearch:
              precision    recall  f1-score   support

           0       0.77      0.84      0.80       100
           1       0.64      0.54      0.59        54

    accuracy                           0.73       154
   macro avg       0.71      0.69      0.69       154
weighted avg       0.73      0.73      0.73       154



In [30]:
comparison_df = pd.DataFrame({
    "Modelo": ["Decision Tree", "RF Base", "RF GridSearch"],
    "Recall": [0.56, 0.59, 0.54],
    "Precision": [0.75, 0.73, 0.64],
    "F1-score": [0.64, 0.65, 0.59]
})

comparison_df.sort_values(by="Recall", ascending=False)

,Modelo,Recall,Precision,F1-score
1,RF Base,0.59,0.73,0.65
0,Decision Tree,0.56,0.75,0.64
2,RF GridSearch,0.54,0.64,0.59


<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

### 💡 **Conclusión probar con hiperparametros**

Tras comparar el rendimiento de los modelos evaluados, se observa que el Random Forest base presenta el mejor recall (0.59), superando ligeramente al Decision Tree (0.56), lo que lo convierte en la mejor opción en este caso, donde es prioritario identificar correctamente los casos positivos.

El Decision Tree, aunque muestra una precisión superior (0.75), queda ligeramente por detrás en términos de recall, lo que implica que deja escapar más casos positivos.

Por otro lado, el modelo optimizado mediante GridSearchCV no logra mejorar el rendimiento del modelo base, obteniendo un recall inferior (0.54). Esto pone de manifiesto que una mayor complejidad o un proceso de optimización más exhaustivo no garantizan necesariamente mejores resultados.

En conjunto, el Random Forest base se selecciona como modelo final por ofrecer el mejor equilibrio entre recall y precisión, alineándose adecuadamente con el objetivo del problema.

</div>

_____________

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3: Guardar el modelo**

</div>

In [32]:
# Guardamos el modelo que nos da el recall mas alto (0.59)
joblib.dump(model_base, '../models/rf_diabetes_final_recall_0.59.sav')

['../models/rf_diabetes_final_recall_0.59.sav']

<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

### 💡 **Conclusión final del proyecto**

En este proyecto se ha comparado el rendimiento de un modelo de árbol de decisión con un modelo Random Forest aplicado al mismo conjunto de datos, utilizando como métrica principal el recall, dada la importancia de identificar correctamente los casos positivos.

El modelo Random Forest base ha demostrado ofrecer el mejor rendimiento en términos de recall, superando ligeramente al Decision Tree y posicionándose como la mejor alternativa para este problema. Esto indica que el uso de modelos ensemble puede aportar mayor capacidad de generalización, aunque no siempre de forma significativa.

Sin embargo, tras explorar distintas combinaciones de hiperparámetros mediante GridSearchCV, se observa que el modelo base ya se encontraba en una configuración cercana al óptimo, ya que las versiones optimizadas no lograron mejorar su rendimiento e incluso lo redujeron en algunos casos.

Este resultado pone de manifiesto que, aunque el uso de modelos más complejos como Random Forest puede aportar mejoras respecto a modelos individuales como el Decision Tree, una optimización adicional mediante ajuste de hiperparámetros no garantiza necesariamente un mejor rendimiento.

En conclusión, este análisis refuerza la importancia de seleccionar modelos y métricas en función del contexto, priorizando la capacidad de detectar casos relevantes frente a métricas globales como la accuracy, y validando siempre las mejoras mediante una evaluación rigurosa.


</div>